In [ ]:
import json
from pathlib import Path

from molmo_spaces.utils.object_metadata import ObjectMeta
from molmo_spaces.molmo_spaces_constants import get_resource_manager, ASSETS_DIR

# Ensure benchmarks are installed
get_resource_manager()

benchmark_dir = ASSETS_DIR / "benchmarks/molmospaces-bench-v2"
benchmark_dir = benchmark_dir.expanduser()

for bench_json in sorted(benchmark_dir.rglob("benchmark.json")):
    if any(ep in str(bench_json) for ep in ["1ep", "10ep", "200ep"]):
        continue

    print(bench_json.relative_to(benchmark_dir))

    with open(bench_json) as f:
        bench_data = json.load(f)

    unique_houses = set([task["house_index"] for task in bench_data])

    if bench_data and "pickup_obj_name" in bench_data[0]["task"]:
        unique_objects = set([task["task"]["pickup_obj_name"] for task in bench_data])
    elif bench_data and "door_body_name" in bench_data[0]["task"]:
        unique_objects = set([task["task"]["door_body_name"] for task in bench_data])
    else:
        print(f"Error occurred while extracting unique objects from {bench_json}")
        continue

    synsets = []
    for name in unique_objects:
        uid = name.split("_")[1]
        anno = ObjectMeta.annotation(uid)
        if anno is None:
            # print(f"No annotation found for {name} UID: {uid}")
            synsets_replacement = name.split("_")[0]
            synsets.append(synsets_replacement)
            continue

        try:
            synsets.append(anno["synset"])
        except KeyError:
            print(f"No synset found for UID: {uid}")

    unique_synsets = set(synsets)

    print("entries:", len(bench_data))
    print("task horizon:", bench_data[0]['task']['task_horizon_sec'])
    print("houses:", len(unique_houses))
    print("objects:", len(unique_objects))
    print("synsets:", len(unique_synsets))
    print()